In [1]:
from embedder import Embedder

embed = Embedder()

query = "How does approximate nearest neighbor search work?"

qv = embed.encode(query)

In [2]:
qv[0]

np.float64(-0.020582034609969147)

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [4]:
page = "02-vector-search/lessons/07-sqlitesearch-vector.md"

In [5]:
doc = [d["content"] for d in documents if d["filename"] == page][0]

In [6]:
docv = embed.encode(doc)
docv.dot(qv)

np.float64(0.3610703009158737)

In [7]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [8]:
from tqdm.auto import tqdm
import numpy as np

X = []
batch_size = 50

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = [c["content"] for c in chunks[i:i + batch_size]]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [9]:
scores = X.dot(qv)

In [10]:
idx = scores.argmax()
chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [11]:
q2 = "What metric do we use to evaluate a search engine?"
vq2 = embed.encode(q2)

In [12]:
from minsearch import VectorSearch

chunk_index = VectorSearch()
chunk_index.fit(X, chunks)
chunk_index.search(vq2, num_results=1)[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [13]:
from minsearch import Index
def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

In [14]:
chunk_text_index = build_index(chunks)

In [15]:
q3 = "How do I store vectors in PostgreSQL?"
vq3 = embed.encode(q3)

In [16]:
top5vec = [c["filename"] for c in chunk_index.search(vq3, num_results=5)]

In [17]:
top5text = [c["filename"] for c in chunk_text_index.search(q3, num_results=5)]

In [18]:
for res in top5vec:
    if res not in top5text:
        print(res)
        break

02-vector-search/lessons/08-pgvector.md


In [19]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [20]:
q4 = "How do I give the model access to tools?"
vq4 = embed.encode(q4)

In [21]:
rrf([chunk_index.search(vq4, num_results=5), chunk_text_index.search(q4, num_results=5)])[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'